💡 **Environment:** `clamp-analyses`  


# Description

**Isolated copy of the parent `10_prediction_performance.ipynb`** for the
`assoc_based_lv_selection` analysis. Reads the drug-disease prediction HDF5 files from the
parent notebooks 06–09 (gene-based and module-based) **plus** this analysis' association-based
variant (`01_prediction_module_based_archs4_assoc`), and computes final performance measures
(AUROC, average precision) against the PharmacotherapyDB gold standard.

All outputs are written under this subfolder — the parent NB 10 outputs are not touched.

Aggregation:
1. Group by (trait, drug, method, tissue) and average ranks across thresholds.
2. Group by (trait, drug, method) and take the max across tissues.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from collections import defaultdict
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score

from pyprojroot import here

# Settings

In [3]:
N_TISSUES = 49
N_THRESHOLDS = 5

METHOD_RENAME = {
    'Gene-based':             'gene_based',
    'Module-based (ARCHS4)':  'module_based_archs4',
    'Module-based':           'module_based_archs4',
    'Module-based (GTEx)':    'module_based_gtex',
    'Module-based (recount2)':'module_based_recount2',
    # association-based LV-selection variant (this isolated analysis)
    'module_based_archs4_assoc': 'module_based_archs4_assoc',
}

METHOD_THRESHOLDS = {
    'gene_based': [-1.0, 50.0, 100.0, 250.0, 500.0],
    'module_based_archs4': [-1.0, 5.0, 10.0, 25.0, 50.0],
    'module_based_gtex': [-1.0, 5.0, 10.0, 25.0, 50.0],
    'module_based_recount2': [-1.0, 5.0, 10.0, 25.0, 50.0],
    'module_based_archs4_assoc': [-1.0, 5.0, 10.0, 25.0, 50.0],
}
EXPECTED_METHODS = tuple(METHOD_THRESHOLDS)
N_PREDICTION_FILES_TOTAL = N_TISSUES * sum(len(v) for v in METHOD_THRESHOLDS.values())

In [4]:
DATA_DIR = here('data/drug_disease_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

# Collect prediction HDF5 files: the 4 parent prediction notebooks (06-09)
# plus this analysis' association-based variant (01).
PREDICTIONS_DIRS = [
    here('output/03_model_biology/00_archs4/02_drug_disease_associations/06_prediction_single_gene_based') / 'lincs' / 'predictions' / 'dotprod_neg',
    here('output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4') / 'lincs' / 'predictions' / 'dotprod_neg',
    here('output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex') / 'lincs' / 'predictions' / 'dotprod_neg',
    here('output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2') / 'lincs' / 'predictions' / 'dotprod_neg',
    here('output/03_model_biology/00_archs4/02_drug_disease_associations/assoc_based_lv_selection/01_prediction_module_based_archs4_assoc') / 'lincs' / 'predictions' / 'dotprod_neg',
]
for d in PREDICTIONS_DIRS:
    display(d)
    assert d.exists(), d

OUTPUT_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/assoc_based_lv_selection/02_prediction_performance') / 'lincs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'predictions').mkdir(parents=True, exist_ok=True)
display(OUTPUT_DIR)

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/data/drug_disease_associations')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/06_prediction_single_gene_based/lincs/predictions/dotprod_neg')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/07_prediction_module_based_archs4/lincs/predictions/dotprod_neg')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/08_prediction_module_based_gtex/lincs/predictions/dotprod_neg')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions/dotprod_neg')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/assoc_based_lv_selection/01_prediction_module_based_archs4_assoc/lincs/predictions/dotprod_neg')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/assoc_based_lv_selection/02_prediction_performance/lincs')

# Load PharmacotherapyDB gold standard

In [5]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())
display(gold_standard['true_class'].value_counts())

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


true_class
1    755
0    243
Name: count, dtype: int64

# Load drug-disease predictions

In [6]:
current_prediction_files = []
for d in PREDICTIONS_DIRS:
    current_prediction_files.extend(sorted(d.glob('*.h5')))
current_prediction_files.sort()
display(len(current_prediction_files))

1225

In [7]:
current_prediction_files[:5]

[PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/06_prediction_single_gene_based/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-data-all_genes-prediction_scores.h5'),
 PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/06_prediction_single_gene_based/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-data-top_100_genes-prediction_scores.h5'),
 PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/06_prediction_single_gene_based/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-data-top_250_genes-prediction_scores.h5'),
 PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/06_prediction_single_gene_based/lincs/predictions/dotprod_neg/spredixca

In [8]:
def _get_tissue(data_value):
    """
    Extracts tissue name from the metadata 'data' field.
    Handles raw data and dataset-specific projected suffixes.
    """
    prefix = 'spredixcan-mashr-zscores-'
    assert data_value.startswith(prefix), data_value
    tissue = data_value[len(prefix):]

    for suffix in (
        '-projection-archs4',
        '-projection-gtex',
        '-projection-recount2',
        '-projection',
        '-data',
    ):
        if tissue.endswith(suffix):
            return tissue[:-len(suffix)]

    raise ValueError(f'Cannot extract tissue from metadata data value: {data_value}')

In [9]:
# Load all prediction files, rank scores, merge with gold standard
predictions = []
skipped_files = []

for f in tqdm(current_prediction_files, ncols=100):
    metadata = pd.read_hdf(f, key='metadata')
    method_name = METHOD_RENAME.get(metadata['method'].values[0], metadata['method'].values[0])
    if method_name not in METHOD_THRESHOLDS:
        skipped_files.append((f.name, method_name))
        continue

    # Load DOID-mapped predictions and rank within the full DOID distribution
    prediction_data = pd.read_hdf(f, key='prediction')
    prediction_data['score'] = prediction_data['score'].rank()

    # Filter to gold-standard pairs
    prediction_data = pd.merge(
        prediction_data, gold_standard, on=['trait', 'drug'], how='inner'
    )
    prediction_data['trait'] = prediction_data['trait'].astype('category')
    prediction_data['drug'] = prediction_data['drug'].astype('category')

    prediction_data = prediction_data.assign(method=method_name)
    prediction_data['method'] = pd.Categorical(
        prediction_data['method'], categories=EXPECTED_METHODS, ordered=True
    )

    prediction_data = prediction_data.assign(n_top_genes=metadata['n_top_genes'].values[0])

    data_value = metadata['data'].values[0]
    prediction_data = prediction_data.assign(data=data_value)
    prediction_data['data'] = prediction_data['data'].astype('category')

    # Extract tissue name from data field
    prediction_data = prediction_data.assign(tissue=_get_tissue(data_value))

    predictions.append(prediction_data)

display(f'Skipped files: {len(skipped_files)}')
if skipped_files:
    display(skipped_files[:10])

100%|███████████████████████████████████████████████████████████| 1225/1225 [03:21<00:00,  6.07it/s]


'Skipped files: 0'

In [10]:
predictions = pd.concat(predictions, ignore_index=True)

In [11]:
display(predictions.shape)
display(predictions.head())

(839125, 8)

,trait,drug,score,true_class,method,n_top_genes,data,tissue
0,DOID:0050741,DB00215,103099.5,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
1,DOID:0050741,DB00704,355210.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
2,DOID:0050741,DB00822,388169.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
3,DOID:10283,DB00014,80190.0,1,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous
4,DOID:10283,DB00175,232448.0,0,gene_based,-1.0,spredixcan-mashr-zscores-Adipose_Subcutaneous-...,Adipose_Subcutaneous


## Validation checks

In [12]:
assert not predictions.isna().any().any()

_method_counts = predictions['method'].value_counts().reindex(EXPECTED_METHODS)
display(_method_counts)

N_PREDICTIONS = predictions[['drug', 'trait']].drop_duplicates().shape[0]
display(f'Unique drug-disease pairs: {N_PREDICTIONS}')

for method_name, thresholds in METHOD_THRESHOLDS.items():
    expected = N_TISSUES * len(thresholds) * N_PREDICTIONS
    actual = int(_method_counts.loc[method_name])
    assert actual == expected, f'{method_name}: expected {expected}, got {actual}'

method
gene_based                   167825
module_based_archs4          167825
module_based_gtex            167825
module_based_recount2        167825
module_based_archs4_assoc    167825
Name: count, dtype: int64

'Unique drug-disease pairs: 685'

In [13]:
_tmp = predictions.groupby(['method', 'n_top_genes'], observed=True).size().rename('n')
display(_tmp)

for method_name, thresholds in METHOD_THRESHOLDS.items():
    method_counts = _tmp.loc[method_name]
    actual_thresholds = sorted(float(x) for x in method_counts.index)
    expected_thresholds = sorted(thresholds)
    assert actual_thresholds == expected_thresholds, (
        f'{method_name}: expected thresholds {expected_thresholds}, got {actual_thresholds}'
    )
    assert np.all(method_counts.values == N_TISSUES * N_PREDICTIONS), method_name

method                     n_top_genes
gene_based                 -1.0           33565
                            50.0          33565
                            100.0         33565
                            250.0         33565
                            500.0         33565
module_based_archs4        -1.0           33565
                            5.0           33565
                            10.0          33565
                            25.0          33565
                            50.0          33565
module_based_gtex          -1.0           33565
                            5.0           33565
                            10.0          33565
                            25.0          33565
                            50.0          33565
module_based_recount2      -1.0           33565
                            5.0           33565
                            10.0          33565
                            25.0          33565
                            50.0          33565
m

## Save raw predictions

In [14]:
output_file = OUTPUT_DIR / 'predictions' / 'predictions_results.pkl'
display(output_file)
predictions.to_pickle(output_file)

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/assoc_based_lv_selection/02_prediction_performance/lincs/predictions/predictions_results.pkl')

# Aggregate predictions

1. Average ranks across n_top_genes thresholds (per trait, drug, method, tissue).
2. Take maximum across tissues (per trait, drug, method).

This matches the PhenoPlier aggregation exactly.

In [15]:
def _reduce_mean(x):
    return pd.Series({
        'score': x['score'].mean(),
        'true_class': x['true_class'].unique()[0],
    })


def _reduce_max(x):
    return pd.Series({
        'score': x['score'].max(),
        'true_class': x['true_class'].unique()[0],
    })

In [16]:
predictions_avg = (
    # Step 1: average across n_top_genes thresholds
    predictions
    .groupby(['trait', 'drug', 'method', 'tissue'], observed=True)
    .apply(_reduce_mean, include_groups=False)
    .dropna()
    # Step 2: take maximum across tissues
    .groupby(['trait', 'drug', 'method'], observed=True)
    .apply(_reduce_max, include_groups=False)
    .dropna()
    .sort_index()
    .reset_index()
)

In [17]:
display(predictions_avg.shape)
display(predictions_avg.head())

assert predictions_avg.shape[0] == len(EXPECTED_METHODS) * N_PREDICTIONS
assert predictions_avg.dropna().shape == predictions_avg.shape

(3425, 5)

,trait,drug,method,score,true_class
0,DOID:0050741,DB00215,gene_based,316134.3,1.0
1,DOID:0050741,DB00215,module_based_archs4,324870.1,1.0
2,DOID:0050741,DB00215,module_based_gtex,349350.5,1.0
3,DOID:0050741,DB00215,module_based_recount2,398655.9,1.0
4,DOID:0050741,DB00215,module_based_archs4_assoc,281332.7,1.0


## Save aggregated predictions

In [18]:
output_file = OUTPUT_DIR / 'predictions' / 'predictions_results_aggregated.pkl'
display(output_file)
predictions_avg.to_pickle(output_file)

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/assoc_based_lv_selection/02_prediction_performance/lincs/predictions/predictions_results_aggregated.pkl')

# ROC performance

In [19]:
# AUROC per method and n_top_genes threshold
predictions.groupby(['method', 'tissue', 'n_top_genes'], observed=True).apply(
    lambda x: roc_auc_score(x['true_class'], x['score']), include_groups=False
).groupby(['method', 'n_top_genes'], observed=True).describe()

count      mean       std       min  \
method                    n_top_genes                                        
gene_based                -1.0          49.0  0.549364  0.023679  0.504653   
                           50.0         49.0  0.537231  0.023829  0.481431   
                           100.0        49.0  0.536558  0.018923  0.487655   
                           250.0        49.0  0.538607  0.023377  0.477309   
                           500.0        49.0  0.540913  0.022241  0.500984   
module_based_archs4       -1.0          49.0  0.547769  0.023010  0.498465   
                           5.0          49.0  0.552505  0.029543  0.482164   
                           10.0         49.0  0.547163  0.028732  0.486371   
                           25.0         49.0  0.549188  0.029233  0.485197   
                           50.0         49.0  0.548572  0.028449  0.488242   
module_based_gtex         -1.0          49.0  0.523350  0.023830  0.473286   
                           5.0          49.0  0.530892  0.028212  0.481198   
                           10.0         49.0  0.532638  0.029691  0.474093   
                           25.0         49.0  0.526181  0.028313  0.470461   
                           50.0         49.0  0.524910  0.028380  0.459455   
module_based_recount2     -1.0          49.0  0.536287  0.020948  0.492913   
                           5.0          49.0  0.545067  0.026516  0.496509   
                           10.0         49.0  0.547962  0.030184  0.488120   
                           25.0         49.0  0.546768  0.021301  0.504042   
                           50.0         49.0  0.544021  0.021914  0.496851   
module_based_archs4_assoc -1.0          49.0  0.547769  0.023010  0.498465   
                           5.0          49.0  0.552895  0.023753  0.500740   
                           10.0         49.0  0.551511  0.024396  0.498600   
                           25.0         49.0  0.538319  0.026578  0.485344   
                           50.0         49.0  0.546358  0.025401  0.489722   

                                            25%       50%       75%       max  
method                    n_top_genes                                          
gene_based                -1.0         0.536803  0.550034  0.564318  0.598705  
                           50.0        0.522263  0.535861  0.552089  0.607265  
                           100.0       0.526958  0.536619  0.550328  0.566788  
                           250.0       0.521321  0.539738  0.556467  0.583908  
                           500.0       0.523131  0.544030  0.556442  0.598693  
module_based_archs4       -1.0         0.532657  0.547845  0.563461  0.591747  
                           5.0         0.536619  0.553312  0.567399  0.617305  
                           10.0        0.526396  0.547845  0.564758  0.620387  
                           25.0        0.529539  0.548493  0.570628  0.633924  
                           50.0        0.527545  0.543015  0.569564  0.612022  
module_based_gtex         -1.0         0.500116  0.526310  0.542685  0.572462  
                           5.0         0.513825  0.526433  0.546145  0.602924  
                           10.0        0.512027  0.528573  0.553752  0.597005  
                           25.0        0.511061  0.528512  0.539542  0.611142  
                           50.0        0.508285  0.523926  0.538209  0.633068  
module_based_recount2     -1.0         0.521651  0.534430  0.552040  0.588286  
                           5.0         0.522862  0.546121  0.563926  0.600160  
                           10.0        0.527399  0.545363  0.567901  0.623016  
                           25.0        0.530713  0.544531  0.561639  0.590940  
                           50.0        0.527704  0.541645  0.560331  0.596944  
module_based_archs4_assoc -1.0         0.532657  0.547845  0.563461  0.591747  
                           5.0         0.533476  0.552260  0.568133  0.618601  
            

In [20]:
# Final AUROC using aggregated predictions
auroc_final = predictions_avg.groupby('method', observed=True).apply(
    lambda x: roc_auc_score(x['true_class'], x['score']), include_groups=False
).rename('AUROC')
display(auroc_final)

method
gene_based                   0.583382
module_based_archs4          0.625419
module_based_gtex            0.602484
module_based_recount2        0.612267
module_based_archs4_assoc    0.574260
Name: AUROC, dtype: float64

These are the final performance measures using AUROC.

# Precision-Recall performance

In [21]:
# Average precision per method and n_top_genes threshold
predictions.groupby(['method', 'tissue', 'n_top_genes'], observed=True).apply(
    lambda x: average_precision_score(x['true_class'], x['score']), include_groups=False
).groupby(['method', 'n_top_genes'], observed=True).describe()

count      mean       std       min  \
method                    n_top_genes                                        
gene_based                -1.0          49.0  0.823048  0.012367  0.791177   
                           50.0         49.0  0.819142  0.011775  0.793504   
                           100.0        49.0  0.819522  0.010505  0.782200   
                           250.0        49.0  0.820481  0.012213  0.788298   
                           500.0        49.0  0.820813  0.011201  0.801469   
module_based_archs4       -1.0          49.0  0.820627  0.013044  0.795488   
                           5.0          49.0  0.823120  0.016803  0.778636   
                           10.0         49.0  0.821277  0.016539  0.779160   
                           25.0         49.0  0.822177  0.016665  0.782468   
                           50.0         49.0  0.821530  0.015951  0.788736   
module_based_gtex         -1.0          49.0  0.808618  0.013861  0.784565   
                           5.0          49.0  0.812254  0.015743  0.774344   
                           10.0         49.0  0.814723  0.016438  0.786304   
                           25.0         49.0  0.812677  0.015250  0.782873   
                           50.0         49.0  0.810747  0.015822  0.769474   
module_based_recount2     -1.0          49.0  0.816358  0.011285  0.789425   
                           5.0          49.0  0.820513  0.013449  0.794799   
                           10.0         49.0  0.824063  0.013886  0.799884   
                           25.0         49.0  0.823404  0.012410  0.787397   
                           50.0         49.0  0.821123  0.011877  0.798447   
module_based_archs4_assoc -1.0          49.0  0.820627  0.013044  0.795488   
                           5.0          49.0  0.812383  0.013354  0.783532   
                           10.0         49.0  0.813348  0.013633  0.787281   
                           25.0         49.0  0.804845  0.013662  0.773560   
                           50.0         49.0  0.812754  0.013112  0.781206   

                                            25%       50%       75%       max  
method                    n_top_genes                                          
gene_based                -1.0         0.817139  0.822989  0.828268  0.845962  
                           50.0        0.811550  0.818870  0.827690  0.850680  
                           100.0       0.812821  0.820163  0.825928  0.837381  
                           250.0       0.812494  0.820298  0.829858  0.841266  
                           500.0       0.814246  0.819441  0.828770  0.849463  
module_based_archs4       -1.0         0.811732  0.820915  0.828317  0.845979  
                           5.0         0.813708  0.820906  0.833490  0.857807  
                           10.0        0.811535  0.819588  0.832004  0.863404  
                           25.0        0.811267  0.820579  0.832461  0.870218  
                           50.0        0.809792  0.820001  0.832301  0.858174  
module_based_gtex         -1.0         0.799159  0.809267  0.820097  0.835931  
                           5.0         0.803583  0.808868  0.823177  0.846560  
                           10.0        0.803776  0.815920  0.825037  0.847027  
                           25.0        0.804159  0.812994  0.822185  0.856536  
                           50.0        0.799408  0.811151  0.821359  0.862883  
module_based_recount2     -1.0         0.807714  0.816788  0.825633  0.837674  
                           5.0         0.812979  0.821179  0.830449  0.845290  
                           10.0        0.816395  0.820898  0.831172  0.860226  
                           25.0        0.816626  0.822205  0.830477  0.850402  
                           50.0        0.811265  0.820492  0.827575  0.847281  
module_based_archs4_assoc -1.0         0.811732  0.820915  0.828317  0.845979  
                           5.0         0.802208  0.809954  0.821875  0.834741  
            

In [22]:
# Final average precision using aggregated predictions
ap_final = predictions_avg.groupby('method', observed=True).apply(
    lambda x: average_precision_score(x['true_class'], x['score']), include_groups=False
).rename('AvgPrecision')
display(ap_final)

method
gene_based                   0.844942
module_based_archs4          0.849646
module_based_gtex            0.839313
module_based_recount2        0.845789
module_based_archs4_assoc    0.825404
Name: AvgPrecision, dtype: float64

These are the final performance measures using average precision.

# Summary

In [23]:
summary = pd.concat([auroc_final, ap_final], axis=1)
display(summary)

,AUROC,AvgPrecision
method,,
gene_based,0.583382,0.844942
module_based_archs4,0.625419,0.849646
module_based_gtex,0.602484,0.839313
module_based_recount2,0.612267,0.845789
module_based_archs4_assoc,0.574260,0.825404
